# 📘 Aula 10f: Processamento de Linguagem Natural com Redes Recorrentes – Embeddings Treináveis & Bidirectional LSTM no IMDB

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula10f%20-%20lstm.ipynb)

Nas aulas anteriores (Aulas 10b), exploramos a abordagem clássica de *Bag-of-Words* (Unigramas, Bigramas e TF-IDF), onde os textos eram transformados em vetores esparsos estáticos sem levar em consideração a ordem estrita das palavras.

Nesta aula, realizamos a grande transição para o **NLP Sequencial Profundo**: trataremos cada resenha de filme como uma **sequência ordenada no tempo de até 600 palavras**. Utilizaremos uma camada de **Word Embedding** treinada do zero para mapear cada índice inteiro de palavra em um vetor geométrico denso contínuo, e uma rede recorrente **Bidirectional LSTM** para ler a frase tanto no sentido direto quanto reverso!

---

### 🎯 Objetivo da Aula
1. Compreender a diferença entre vetorização estática (Bag-of-Words) e **vetorização sequencial ordenada** (`output_mode='int'` com `output_sequence_length=600`).
2. Aprender o conceito de **Word Embedding (Espaço Semântico Vetorial)**: como palavras são projetadas em vetores densos onde distâncias e ângulos refletem similaridade semântica.
3. Entender a importância do parâmetro **`mask_zero=True`**: ensinar a rede recorrente a ignorar automaticamente os tokens de preenchimento (*padding zeros*).
4. Construir uma arquitetura com **`Bidirectional(LSTM)`**, lendo a narrativa simultaneamente do início ao fim e do fim ao início.
5. Treinar, persistir o modelo com `ModelCheckpoint` e mensurar o ganho de acurácia em relação aos modelos não sequenciais.

---

### 💡 A Intenção Pedagógica do Professor
* **Preservação da Sintaxe e Ordem**: Demonstrar que frases como *"not good, absolutely boring"* e *"not boring, absolutely good"* geram o mesmo vetor em um Bag-of-Words tradicional, mas são interpretadas de formas totalmente opostas por uma rede recorrente LSTM!
* **O Poder do `mask_zero`**: Evitar o erro comum de gastar capacidade computacional fazendo a LSTM processar dezenas de zeros vazios em textos curtos.

---

### 📚 Glossário da Aula
* **Word Embedding**: Mapeamento onde palavras com significados semelhantes são representadas por vetores numéricos próximos no espaço latente.
* **Padding (Preenchimento)**: Adição de zeros ao final de sequências menores que `max_length` para uniformizar o formato dos tensores em lotes.
* **Masking**: Mecanismo que instrui camadas subsequentes a ignorar valores de padding durante os cálculos de perda e retropropagação.
* **Bidirectional LSTM**: Duas redes LSTM operando em paralelo (uma lendo na ordem cronológica e outra na ordem reversa), concatenando suas saídas.

---

### 🧭 Trilha da Aula
1. [Download e Extração do Corpus IMDB](#1-download-e-extração-do-dataset-imdb)
2. [Limpeza do Diretório Não Supervisionado](#2-remoção-da-pasta-unsup)
3. [Inspeção de Amostra Real Positiva](#3-inspeção-de-texto-real)
4. [Estruturação dos Subsets com pathlib](#4-particionamento-de-validação)
5. [Carregamento com text_dataset_from_directory](#5-streaming-com-text_dataset_from_directory)
6. [Vetorização Sequencial Inteira (Sequências de 600 Tokens)](#6-vetorização-sequencial-com-textvectorization)
7. [Arquitetura: Embedding Treinável + Bidirectional LSTM + Dropout](#7-arquitetura-com-embedding-e-bidirectional-lstm)
8. [Treinamento e Avaliação no Teste](#8-treinamento-e-avaliação-no-teste)


## 1. As Perguntas Centrais de NLP

* *"Qual é o tópico deste texto?"* (classificação)
* *"Este texto contém ofensas?"* (moderação)
* *"Este comentário expressa sentimento positivo ou negativo?"* **(Análise de Sentimentos)**


## 2. Download e Extração do Dataset IMDB

**O que estamos fazendo**: Baixar e descompactar o arquivo `aclImdb_v1.tar.gz` do Stanford AI Lab.

**Por que fazer assim**: Base com 50.000 resenhas de cinema altamente descritivas para treino e teste.



In [ ]:
# Baixa e extrai o acervo IMDB
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz


## 3. Remoção da Pasta Não Supervisionada

**O que estamos fazendo**: Excluir os dados sem anotação em `aclImdb/train/unsup`.

**Por que fazer assim**: Mantém apenas as pastas de classificação binária `pos` e `neg`.



In [ ]:
# Exclui resenhas não anotadas
!rm -r aclImdb/train/unsup


## 4. Inspeção de Texto Real

**O que estamos fazendo**: Exibir uma resenha positiva anotada com nota 10/10.

**Por que fazer assim**: Mostra a riqueza de adjetivos e a extensão típica de textos livres.



In [ ]:
# Visualiza o conteúdo do arquivo 4077_10.txt
!cat aclImdb/train/pos/4077_10.txt


## 5. Particionamento de Validação com pathlib

**O que estamos fazendo**: Mover aleatoriamente 20% das resenhas de treino para a pasta `val` com semente fixa (1337).

**Por que fazer assim**: Garante 5.000 amostras independentes para validação e salvamento de checkpoints sem tocar no teste.



In [ ]:
import os, pathlib, shutil, random

base_dir = pathlib.Path('aclImdb')
val_dir = base_dir / 'val'
train_dir = base_dir / 'train'

for category in ('neg', 'pos'):
  os.makedirs(val_dir / category, exist_ok=True)
  files = os.listdir(train_dir / category)
  random.Random(1337).shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname,
                val_dir / category / fname)
print('Diretório de validação criado com sucesso!')


## 6. Streaming com text_dataset_from_directory

**O que estamos fazendo**: Instanciar geradores de lotes de 32 resenhas para treino, validação e teste.

**Por que fazer assim**: Evita carregar o texto completo na memória RAM de uma vez.



In [ ]:
from tensorflow import keras
batch_size = 32

print('Carregando partição de treino...')
train_ds = keras.utils.text_dataset_from_directory('aclImdb/train', batch_size=batch_size)
print('Carregando partição de validação...')
val_ds = keras.utils.text_dataset_from_directory('aclImdb/val', batch_size=batch_size)
print('Carregando partição de teste...')
test_ds = keras.utils.text_dataset_from_directory('aclImdb/test', batch_size=batch_size)


## 7. Inspeção dos Lotes Brutos

**O que estamos fazendo**: Verificar formato dos tensores de strings e rótulos de classe.

**Por que fazer assim**: Cada lote contém 32 resenhas de comprimento variável.



In [ ]:
for inputs, targets in train_ds:
  print('Formato do lote de strings:  ', inputs.shape)
  print('Formato do lote de rótulos:   ', targets.shape)
  print('Primeiro texto bruto:\n', inputs[0].numpy().decode('utf-8')[:250], '...')
  print('Rótulo supervisionado:        ', targets[0].numpy())
  break


## 8. Vetorização Sequencial Inteira com TextVectorization

**O que estamos fazendo**: Configurar a camada com `output_mode='int'` e `output_sequence_length=600`, limitando ao vocabulário de 20.000 tokens.

**Por que fazer assim**: Diferente do Bag-of-Words, agora a ordem das palavras é estritamente preservada! Cada resenha vira uma lista sequencial de até 600 inteiros.

> [!NOTE]
> Resenhas com mais de 600 palavras são truncadas; resenhas menores recebem zeros no final (*padding*).



In [ ]:
from tensorflow.keras.layers import TextVectorization

# Limites: 600 palavras por texto e vocabulário máximo de 20.000 tokens
max_length = 600
max_tokens = 20000

# Camada configurada para saída inteira sequencial de tamanho padronizado 600
text_vectorization = TextVectorization(max_tokens=max_tokens,
                                       output_mode='int',
                                       output_sequence_length=max_length)

# Ajusta o vocabulário sobre os textos de treino
text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)

# Mapeia a vetorização em paralelo nas 3 partições
binary_train_ds = train_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
binary_val_ds = val_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
binary_test_ds = test_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)


## 9. Inspeção dos Tensores Sequenciais Finais

**O que estamos fazendo**: Verificar as dimensões do lote após a transformação inteira.

**Por que fazer assim**: O tensor possui formato `(32, 600)`: 32 amostras e exatamente 600 posições de palavras consecutivas.



In [ ]:
for inputs, targets in binary_train_ds:
  print('Formato do lote sequencial (amostras, passos_temporais):', inputs.shape)
  print('Primeiros 30 tokens inteiros da primeira resenha:\n', inputs[0].numpy()[:30])
  break


## 10. Arquitetura com Embedding e Bidirectional LSTM

**O que estamos fazendo**: Montar a rede com: Entrada (600 inteiros) -> `Embedding(20000, 64, mask_zero=True)` -> `Bidirectional(LSTM(32))` -> `Dropout(0.5)` -> `Dense(1, sigmoid)`.

**Por que fazer assim**: A camada de Embedding converte inteiros em vetores de 64 dimensões que aprendem a semântica das palavras durante o treino. A BiLSTM processa a sequência nos dois sentidos cronológicos.

> [!TIP]
> `mask_zero=True` propaga uma máscara booleana que instrui a LSTM a pular os zeros de preenchimento, economizando computação e evitando ruído!



In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def get_model(max_tokens=20000, hidden_dim=32):
  """
  Constrói rede neural sequencial profunda com Embedding treinável e BiLSTM.
  """
  inputs = keras.Input(shape=(None,), dtype='int64')
  # Camada de Embedding com mascaramento de zeros de preenchimento
  embedded = layers.Embedding(input_dim=max_tokens,
                              output_dim=64,
                              mask_zero=True)(inputs)
  # LSTM Bidirecional: lê a sequência de palavras do início ao fim e do fim ao início
  x = layers.Bidirectional(layers.LSTM(hidden_dim))(embedded)
  # Dropout de 50% para combater sobreajuste
  x = layers.Dropout(0.5)(x)
  # Saída sigmoide para probabilidade de sentimento positivo
  outputs = layers.Dense(1, activation='sigmoid')(x)
  
  model = keras.Model(inputs, outputs)
  model.compile(optimizer='rmsprop',
                loss='binary_crossentropy',
                metrics=['accuracy'])
  return model

model = get_model()
model.summary()


## 11. Treinamento e Avaliação no Teste

**O que estamos fazendo**: Treinar a rede por 10 épocas salvando o melhor checkpoint `lstm.keras` e avaliar nos 25.000 textos de teste.

**Por que fazer assim**: Mensura a capacidade de generalização da arquitetura sequencial profunda.



In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint('lstm.keras', save_best_only=True)
]

history = model.fit(binary_train_ds,
                    validation_data=binary_val_ds,
                    epochs=10,
                    callbacks=callbacks)

# Avalia o melhor checkpoint salvo no teste
model = keras.models.load_model('lstm.keras')
test_loss, test_acc = model.evaluate(binary_test_ds)
print(f'Perda no Teste (LSTM):    {test_loss:.3f}')
print(f'Acurácia no Teste (LSTM): {test_acc * 100:.2f}%')


---

## 🎓 Resumo da Aula & O Que Você Aprendeu

Nesta aula, dominamos a transição para modelos neurais sequenciais em NLP:

1. **Preservação Sequencial**: Passamos a vetorizar textos como sequências temporais de 600 inteiros (`output_mode='int'`).
2. **Word Embeddings**: Aprendemos como a camada `Embedding` projeta índices discretos em um espaço semântico contínuo e denso.
3. **Mascaramento com `mask_zero=True`**: Protegemos a rede recorrente contra os efeitos do preenchimento com zeros.
4. **Bidirectional LSTM**: Combinamos duas passagens recorrentes temporais para capturar o contexto completo da frase.

---

## 👉 Próximo Passo
Na **Aula 10g**, aprenderemos como potencializar ainda mais esse modelo utilizando **Embeddings Pré-treinados GloVe (Stanford)**, trazendo para o NLP a mesma revolução do Transfer Learning que vimos na Visão Computacional!
